# 03.2 LSTM 入门（LSTM Intro）

`LSTM` 是经典序列模型之一。  

即使现在很多场景都转向 Transformer，`LSTM` 依然有很高的学习价值，因为它能帮助你理解：  

- 序列信息是如何按时间步传播的
- 隐状态（hidden state）
- 循环模型输入输出的 shape

本 notebook 会用一个很小的合成任务来演示 `LSTM` 的完整流程。  


## 学习目标

学完后你应该能

1. 理解 `LSTM` 的输入输出 shape
2. 理解 `hidden state` 和 `cell state`
3. 写一个最小 `LSTM` 分类模型
4. 看懂 `batch_first=True` 的意义
5. 在一个小序列任务上训练并评估 `LSTM`
6. Transformer 学习建立对比参照

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## 1. 先看一个最小 `LSTM` 的 shape

先不训练，先把 shape 看清楚。  

这里我们使用 `batch_first=True`，所以输入 shape 是：  

- `(batch_size, seq_len, input_size)`

In [ ]:
lstm = nn.LSTM(input_size=5, hidden_size=7, batch_first=True)
x = torch.randn(4, 6, 5)
output, (h_n, c_n) = lstm(x)

print("x.shape =", x.shape)
print("output.shape =", output.shape)
print("h_n.shape =", h_n.shape)
print("c_n.shape =", c_n.shape)

这里的含义

- `output.shape == (4, 6, 7)`
  time step 都有一个 hidden representation
- `h_n.shape == (1, 4, 7)`
  最终 hidden state
- `c_n.shape == (1, 4, 7)`
  最终 cell state

## 2. 构造一个小任务

为了让 `LSTM` 真正学到“序列关系”，这里构造一个玩具任务：  

- 输入是长度为 6 的整数序列
- 标签是“首 token 是否等于尾 token”

这个任务需要模型记住开头信息，所以比单纯看某一个位置更像序列任务。  


In [ ]:
torch.manual_seed(0)

vocab_size = 8
seq_len = 6
num_samples = 1000

X = torch.randint(low=0, high=vocab_size, size=(num_samples, seq_len))
y = (X[:, 0] == X[:, -1]).long()

split = 800
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

print("positive rate / 正样本比例:", y.float().mean().item())
print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)

## 3. 定义 `LSTM` 分类模型

常见流程

1. token ids -> `Embedding`
2. `Embedding` -> `LSTM`
3. 取最终 hidden state -> 分类头

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        output, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]
        logits = self.fc(last_hidden)
        return logits


model = LSTMClassifier(vocab_size=vocab_size, embed_dim=12, hidden_size=16, num_classes=2)
print(model)

In [ ]:
xb, yb = next(iter(train_loader))
logits = model(xb)

print("xb.shape =", xb.shape)
print("logits.shape =", logits.shape)
print("yb.shape =", yb.shape)

这里 `logits.shape == (batch_size, 2)`，因为这是一个二分类任务，但我们用 2 个类别分数来表示。  


## 4. 训练与评估函数

这里延续你前面已经熟悉的训练套路。  


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


def batch_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## 5. 开始训练

这里训练 8 个 epoch，足够看见 `LSTM` 学到这个小任务。  


In [ ]:
history = []

for epoch in range(1, 9):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    print(
        f"epoch={epoch:02d} | "
        f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

In [ ]:
print(history[-1])

## 6. 观察预测

这里直接看看模型对几条序列的预测。  


In [ ]:
model.eval()
sample_x = X_val[:8]
sample_y = y_val[:8]

with torch.no_grad():
    sample_logits = model(sample_x)
    sample_preds = sample_logits.argmax(dim=1)

print("sample_x =\n", sample_x)
print("true labels =", sample_y)
print("pred labels =", sample_preds)

## 7. 一个 shape 练习

现在你要开始熟悉 `LSTM` 的三个关键 shape：  

- 输入（input）
- 全部时间步输出（all-step outputs）
- 最终 hidden state（final hidden state）

In [ ]:
# 练习 1
# 已知
# batch_size = 5
# seq_len = 7
# embed_dim = 12
# hidden_size = 16
# batch_first=True
#
# 问题
# 1. 输入到 LSTM 的 x.shape 是多少？
# 2. output.shape 是多少？
# 3. h_n.shape 是多少？
#
# 请先自己回答，再看参考答案。
# Answer first by yourself, then check the reference answer.

参考答案

- `x.shape == (5, 7, 12)`
- `output.shape == (5, 7, 16)`
- `h_n.shape == (1, 5, 16)`

这里默认是单层单向 `LSTM`。  


In [ ]:
# 练习 2
# 把模型里的 hidden_size 从 16 改成 32，再观察 logits.shape 是否变化。
# Change the model's hidden_size from 16 to 32, then observe whether logits.shape changes.

提示

`logits.shape` 最终由输出类别数决定，而不是直接由 hidden size 决定。  


## 8. 小结

本节最重要的是先把 `LSTM` 的 shape 和信息流看清楚。  

你现在应该能回答

1. `LSTM` 输入 shape 为什么常写成 `(B, T, D)`？
2. `output` 和 `h_n` 有什么区别？
3. 为什么分类任务常用最后 hidden state 接分类头？

下一步建议

- 进入 `Attention` notebook，对比循环模型和注意力模型的处理方式（Move to the `Attention` notebook and compare recurrent models with attention-based models.）